In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
os.getcwd()

'/Users/devanteheywood/repayment-risk-simulator/notebooks'

In [3]:
os.listdir("../data/raw")

['accepted_2007_to_2018Q4.csv']

In [4]:
df = pd.read_csv("../data/raw/accepted_2007_to_2018Q4.csv", low_memory=False)

In [5]:
df.shape

(2260701, 151)

In [6]:
df.head()

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,NaN,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,68355089,NaN,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,68341763,NaN,20000.0,20000.0,20000.0,60 months,10.78,432.66,B,B4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,66310712,NaN,35000.0,35000.0,35000.0,60 months,14.85,829.90,C,C5,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,68476807,NaN,10400.0,10400.0,10400.0,60 months,22.45,289.91,F,F1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
df["loan_status"].value_counts()

loan_status
Fully Paid                                             1076751
Current                                                 878317
Charged Off                                             268559
Late (31-120 days)                                       21467
In Grace Period                                           8436
Late (16-30 days)                                         4349
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     40
Name: count, dtype: int64

In [8]:
bad_status = [
    "Charged Off",
    "Default",
    "Late (31-120 days)",
    "Late (16-30 days)"
]

df["risk_flag"] = df["loan_status"].isin(bad_status).astype(int)

/var/folders/p3/gtr_z1n149s0bdjhftl66z4w0000gn/T/ipykernel_68259/286087547.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["risk_flag"] = df["loan_status"].isin(bad_status).astype(int)


In [9]:
df = df[df["loan_status"].isin(bad_status + ["Fully Paid"])]

In [11]:
df["risk_flag"].mean()

np.float64(0.21471871385375657)

In [12]:
features = [
    "loan_amnt",
    "term",
    "int_rate",
    "grade",
    "emp_length",
    "home_ownership",
    "annual_inc",
    "dti",
    "fico_range_low",
    "fico_range_high",
    "purpose",
    "risk_flag"
]

model_df = df[features].copy()

In [14]:
# Ensure interest rate is numeric
model_df["int_rate"] = pd.to_numeric(model_df["int_rate"], errors="coerce")

In [15]:
model_df["term"] = model_df["term"].str.extract("(\d+)").astype(int)

In [16]:
model_df["fico_avg"] = (
    model_df["fico_range_low"] + model_df["fico_range_high"]
) / 2

model_df.drop(columns=["fico_range_low", "fico_range_high"], inplace=True)

In [17]:
model_df.head()
model_df.shape

(1371166, 11)

In [18]:
model_df.to_csv("../data/processed/bi_ready_dataset.csv", index=False)

In [19]:
import os
os.listdir("../data/processed")

['bi_ready_dataset.csv']